<a href="https://colab.research.google.com/github/sean838432/TdAI/blob/main/data_download/ASOS_download.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
This script grabs dewpoint data from specified ASOS stations and saves the data to CSV files

Info on CGI parameters is available at:

    https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?help
"""

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import requests
import os
import datetime
import time
import io
import pandas as pd

Mounted at /content/drive


In [ ]:
################################## INPUTS ####################################
# List of stations and their corresponding IEM networks
STATIONS = ['CAR'] #, 'GNR', 'CON', 'BDL', 'ALB', 'BGM']
NETWORK_MAP = {'CAR': 'ME_ASOS'} #, 'GNR': 'ME_ASOS', 'CON': 'NH_ASOS', 'BDL': 'CT_ASOS', 'ALB': 'NY_ASOS', 'BGM': 'NY_ASOS'}

DATA_VARS = ['dwpf', 'valid']
BASE_URL = 'https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py'

# Date range for the bulk request
START_YEAR = 2020
END_YEAR = 2026

# Seasonal Filtering (March 1 to Nov 15)
START_MD = (3, 1)
END_MD = (11, 15)

SAVE_DIR = "/content/drive/MyDrive/Fire Weather Focal Point/TdAI_Project/Development/Data/"
##############################################################################

if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

for station in STATIONS:
    network = NETWORK_MAP.get(station)
    if not network:
        print(f"⚠️ Network not found for {station}. Skipping.")
        continue

    params = {
        'station': station,
        'network': network,
        'data': DATA_VARS,
        'year1': START_YEAR, 'month1': 1, 'day1': 1,
        'year2': END_YEAR, 'month2': 12, 'day2': 31,
        'format': 'onlycomma',
        'tz': 'UTC',
        'missing': 'M',
        'report_type': 3, # METAR reports only
    }

    try:
        print(f"🚀 Fetching bulk data for {station} ({START_YEAR}-{END_YEAR})...")
        response = requests.get(BASE_URL, params=params, timeout=60)
        response.raise_for_status()

        # Load into Pandas for fast seasonal filtering
        df = pd.read_csv(
            io.StringIO(response.text),
            comment='#',
            skiprows=1,
            names=['Station', 'valid_time', 'ASOS Dewpoint (F)'],
            na_values='M'
        )

        initial_count = len(df)
        df = df.dropna(subset=['ASOS Dewpoint (F)'])
        dropped_count = initial_count - len(df)

        if dropped_count > 0:
            print(f"🧹 Cleaned up {dropped_count} missing ('M') observations.")

        if df.empty:
            print(f"⚠️ No data remaining for {station} after cleaning.")
            continue

        # Convert to datetime with explicit format
        df['valid_time'] = pd.to_datetime(df['valid_time'], format='%Y-%m-%d %H:%M')

        # --- THE SEASONAL FILTER ---
        # We create a mask for the month/day window across all years
        df['month_day'] = df['valid_time'].apply(lambda x: (x.month, x.day))
        seasonal_mask = (df['month_day'] >= START_MD) & (df['month_day'] <= END_MD)
        df_filtered = df[seasonal_mask].drop(columns=['month_day'])

        # Save individual file
        filename = f"K{station}_{START_YEAR}_to_{END_YEAR}_asos.csv"
        save_path = os.path.join(SAVE_DIR, filename)

        df_filtered.to_csv(save_path, index=False)
        print(f"✅ Saved {len(df_filtered)} obs for K{station} to {filename}")

        # Wait to be polite to the server before moving to next station
        print("⏳ Waiting 10 seconds before next station...")
        time.sleep(10)

    except Exception as e:
        print(f"❌ Error for {station}: {e}")
        time.sleep(30) # Longer wait on error

print("\n✨ All station downloads complete!")

🚀 Fetching bulk data for CAR (2020-2026)...
🧹 Cleaned up 100 missing ('M') observations.
✅ Saved 39414 obs for KCAR to KCAR_2020_to_2026_asos.csv
⏳ Waiting 10 seconds before next station...

✨ All station downloads complete!
